# HGCD — Hop-Graph Contrastive Decoding (Kaggle run-all)

Training-free, inference-time causal intervention for multi-hop reasoning.

**Setup on Kaggle:** *Settings → Accelerator → GPU T4 x2*, *Settings → Internet → On*, then **Run All**.

What this notebook does (all numbers it prints are produced here, nothing is hard-coded):
1. **Estimator validation** — gold chains + one inserted true-but-redundant hop (true necessity 0):
   AUC of the graph-aware total-effect estimator vs. naive masking vs. placebo (CLUTRR and MuSiQue).
2. **Alpha selection** on the CLUTRR *validation* split (never on test).
3. **CLUTRR test** (ID k≤4, OOD k≥5) with binary interventions `rename / order / distract / placebo_input`
   (true effect on the answer = 0) → accuracy + paired ATE with bootstrap CIs.
4. **MuSiQue** (real text, bridge-entity chains) with `dist8 / order` interventions.
5. Writes LaTeX tables (`tab_*.tex`), figures, CSVs and `summary.json` into `/kaggle/working/results/`.

GPU 0 runs the main model; GPU 1 (optional cell) runs a second model family in parallel.


In [ ]:
!pip -q install -U "transformers>=4.44" accelerate scikit-learn 2>/dev/null | tail -1
!nvidia-smi --query-gpu=name,memory.total --format=csv || true

In [ ]:
import os
os.makedirs('hgcd', exist_ok=True)
os.environ.setdefault('HGCD_DATA', os.path.abspath('hgcd_data'))

In [ ]:
%%writefile hgcd/__init__.py
"""HGCD: Hop-Graph Contrastive Decoding for multi-hop reasoning.

Training-free, inference-time interventions on a model's own reasoning graph.
"""

__version__ = "0.1.0"


In [ ]:
%%writefile hgcd/data.py
"""Dataset loading and controlled (binary) interventions.

CLUTRR  : kinship chains with a gold path graph (controlled domain).
MuSiQue : real multi-hop QA with gold decompositions (bridge-entity chains).

Every intervention is a deterministic, seeded program (no LLM involved), so the
ground-truth effect of each intervention is known by construction:
  rename / order / distract / placebo_input  ->  true effect on the answer = 0.
"""

from __future__ import annotations

import ast
import json
import os
import random
import re
import urllib.request
from dataclasses import dataclass, field, replace
from typing import Dict, List, Optional, Tuple

CLUTRR_URL = ("https://raw.githubusercontent.com/kliang5/CLUTRR_huggingface_dataset/"
              "main/{task}/{split}.csv")
CLUTRR_TASK = "gen_train234_test2to10"

# Fictional names (not used by CLUTRR) for renaming / distractor families.
FRESH_NAMES = {
    "male": ["Orrin", "Thadeus", "Brannock", "Celwyn", "Doran", "Eldric", "Fennick", "Garrow",
             "Halvard", "Ivor", "Jorvik", "Kestrel", "Lorcan", "Madoc", "Norris", "Osric",
             "Perrin", "Quillon", "Roderick", "Soren", "Tobias", "Ulric", "Varek", "Wystan",
             "Yorick", "Zander", "Alaric", "Bastian", "Corwin", "Dashiel"],
    "female": ["Aveline", "Brisa", "Calla", "Delphine", "Elowen", "Fenna", "Giselle", "Hesper",
               "Isolde", "Junia", "Kerensa", "Liora", "Maelis", "Nerys", "Oriel", "Perpetua",
               "Rosalind", "Sabine", "Talitha", "Undine", "Verity", "Winsome", "Xanthe", "Yseult",
               "Zinnia", "Amabel", "Briallen", "Cressida", "Damaris", "Evadne"],
}

NEUTRAL_SENTENCES = [
    "It was a sunny afternoon.",
    "The town had a small library near the river.",
    "Everyone agreed that the soup was delicious.",
    "The bus arrived a few minutes late that day.",
    "A light rain fell during the evening.",
]


# ----------------------------------------------------------------------------- utils

def _cache_dir() -> str:
    d = os.environ.get("HGCD_DATA", os.path.join(os.path.expanduser("~"), ".cache", "hgcd"))
    os.makedirs(d, exist_ok=True)
    return d


def _download(url: str, path: str) -> str:
    if not os.path.exists(path):
        urllib.request.urlretrieve(url, path)
    return path


def strip_brackets(text: str) -> str:
    return text.replace("[", "").replace("]", "")


def split_sentences(text: str) -> List[str]:
    parts = re.split(r"(?<=[.!?])\s+", text.strip())
    return [p for p in parts if p]


def _sub_names(text: str, mapping: Dict[str, str]) -> str:
    """Replace bracketed CLUTRR names ([Name]) according to mapping."""
    def rep(m):
        n = m.group(1)
        return "[" + mapping.get(n, n) + "]"
    return re.sub(r"\[([^\]]+)\]", rep, text)


# ----------------------------------------------------------------------------- CLUTRR

@dataclass
class GoldHop:
    head: str
    rel: str
    tail: str

    def text(self) -> str:
        return f"{self.head}'s {self.rel} is {self.tail}."


@dataclass
class ClutrrExample:
    uid: str
    story: str                    # raw story with [Name] brackets
    names: List[str]              # entity names, index order
    genders: Dict[str, str]
    source: str                   # query[0]
    target: str                   # query[1]
    answer: str                   # relation: target is source's <answer>
    gold_hops: List[GoldHop]
    condition: str = "orig"
    extra_facts: List[GoldHop] = field(default_factory=list)  # true facts off the path

    @property
    def k(self) -> int:
        return len(self.gold_hops)

    @property
    def story_text(self) -> str:
        return strip_brackets(self.story)

    def question(self) -> str:
        return f"How is {self.target} related to {self.source}?"

    def answer_prefix(self) -> str:
        return f"Answer: {self.target} is {self.source}'s"


def _parse_clutrr_row(row: dict) -> Optional[ClutrrExample]:
    edges = ast.literal_eval(row["story_edges"])
    qedge = ast.literal_eval(row["query_edge"])
    if edges != [(i, i + 1) for i in range(len(edges))] or qedge != (0, len(edges)):
        return None  # keep only clean simple-path stories (documented filter)
    rels = ast.literal_eval(row["edge_types"])
    gpairs = [g.split(":") for g in row["genders"].split(",")]
    names = [g[0] for g in gpairs]
    genders = {g[0]: g[1] for g in gpairs}
    if len(names) != len(edges) + 1:
        return None
    hops = [GoldHop(names[i], rels[i], names[i + 1]) for i in range(len(edges))]
    return ClutrrExample(uid=row["id"], story=row["story"], names=names, genders=genders,
                         source=names[0], target=names[-1], answer=row["target_text"],
                         gold_hops=hops)


def load_clutrr(split: str = "test", task: str = CLUTRR_TASK) -> List[ClutrrExample]:
    import csv
    path = os.path.join(_cache_dir(), f"clutrr_{task}_{split}.csv")
    _download(CLUTRR_URL.format(task=task, split=split), path)
    out = []
    with open(path, newline="", encoding="utf-8") as f:
        for row in csv.DictReader(f):
            ex = _parse_clutrr_row(row)
            if ex is not None:
                out.append(ex)
    return out


def clutrr_relations(examples: List[ClutrrExample]) -> List[str]:
    rels = set()
    for ex in examples:
        rels.add(ex.answer)
        rels.update(h.rel for h in ex.gold_hops)
    return sorted(rels)


def sample_by_k(examples: List[ClutrrExample], ks: List[int], n_per_k: int,
                seed: int = 0) -> List[ClutrrExample]:
    rng = random.Random(seed)
    out = []
    for k in ks:
        pool = [e for e in examples if e.k == k]
        rng.shuffle(pool)
        out.extend(pool[:n_per_k])
    return out


# --- interventions (all binary, seeded, ground-truth effect on the answer = 0)

def _fresh_mapping(names: List[str], genders: Dict[str, str], rng: random.Random,
                   avoid: Optional[set] = None) -> Dict[str, str]:
    avoid = set(avoid or ())
    pools = {g: [n for n in FRESH_NAMES[g] if n not in avoid] for g in FRESH_NAMES}
    for g in pools:
        rng.shuffle(pools[g])
    mapping = {}
    for n in names:
        g = genders.get(n, "male")
        mapping[n] = pools[g].pop() if pools[g] else n + "x"
    return mapping


def rename(ex: ClutrrExample, seed: int = 0) -> ClutrrExample:
    rng = random.Random(f"rename-{ex.uid}-{seed}")
    m = _fresh_mapping(ex.names, ex.genders, rng)
    return replace(ex, story=_sub_names(ex.story, m), names=[m[n] for n in ex.names],
                   genders={m[n]: g for n, g in ex.genders.items()},
                   source=m[ex.source], target=m[ex.target],
                   gold_hops=[GoldHop(m[h.head], h.rel, m[h.tail]) for h in ex.gold_hops],
                   condition="rename")


def reorder(ex: ClutrrExample, seed: int = 0) -> ClutrrExample:
    rng = random.Random(f"order-{ex.uid}-{seed}")
    sents = split_sentences(ex.story)
    if len(sents) > 1:
        orig = list(sents)
        for _ in range(10):
            rng.shuffle(sents)
            if sents != orig:
                break
    return replace(ex, story=" ".join(sents), condition="order")


def _distractor_family(ex: ClutrrExample, pool: List[ClutrrExample], rng: random.Random):
    """A disconnected family from another story, renamed with fresh names."""
    other = rng.choice([o for o in pool if o.uid != ex.uid and o.k <= 3] or pool)
    m = _fresh_mapping(other.names, other.genders, rng, avoid=set(ex.names))
    story = _sub_names(other.story, m)
    facts = [GoldHop(m[h.head], h.rel, m[h.tail]) for h in other.gold_hops]
    return story, facts, {m[n]: g for n, g in other.genders.items()}


def distract(ex: ClutrrExample, pool: List[ClutrrExample], seed: int = 0) -> ClutrrExample:
    rng = random.Random(f"distract-{ex.uid}-{seed}")
    dstory, facts, dgen = _distractor_family(ex, pool, rng)
    sents = split_sentences(ex.story)
    for s in split_sentences(dstory):
        sents.insert(rng.randint(0, len(sents)), s)
    g = dict(ex.genders)
    g.update(dgen)
    return replace(ex, story=" ".join(sents), genders=g, extra_facts=facts, condition="distract")


def placebo_input(ex: ClutrrExample, seed: int = 0) -> ClutrrExample:
    rng = random.Random(f"placebo-{ex.uid}-{seed}")
    sents = split_sentences(ex.story)
    for s in rng.sample(NEUTRAL_SENTENCES, 2):
        sents.insert(rng.randint(0, len(sents)), s)
    return replace(ex, story=" ".join(sents), condition="placebo_input")


def counterfactual_story(ex: ClutrrExample, seed: int = 0) -> str:
    """Type-matched counterfactual context for L1 contrast.

    Same sentences, same length, same entity *set* and genders, but names are permuted
    within each gender group (cyclic shift), so who-is-related-to-whom changes. Singleton
    gender groups are mapped to a fresh name of the same gender.
    """
    rng = random.Random(f"cf-{ex.uid}-{seed}")
    groups: Dict[str, List[str]] = {}
    for n in ex.names:
        groups.setdefault(ex.genders.get(n, "male"), []).append(n)
    m: Dict[str, str] = {}
    for g, ns in groups.items():
        if len(ns) >= 2:
            sh = list(ns)
            rng.shuffle(sh)
            for a, b in zip(sh, sh[1:] + sh[:1]):
                m[a] = b
        else:
            m[ns[0]] = rng.choice([x for x in FRESH_NAMES[g] if x not in ex.names])
    return strip_brackets(_sub_names(ex.story, m))


def apply_condition(ex: ClutrrExample, cond: str, pool: List[ClutrrExample],
                    seed: int = 0) -> ClutrrExample:
    if cond == "orig":
        return ex
    if cond == "rename":
        return rename(ex, seed)
    if cond == "order":
        return reorder(ex, seed)
    if cond == "distract":
        return distract(ex, pool, seed)
    if cond == "placebo_input":
        return placebo_input(ex, seed)
    raise ValueError(cond)


def with_redundant_fact(ex: ClutrrExample, pool: List[ClutrrExample],
                        seed: int = 0) -> Tuple[ClutrrExample, GoldHop]:
    """Add a disconnected family to the story and return one true-but-irrelevant fact.

    Used to build gold chains with an inserted redundant hop whose true necessity is 0.
    """
    ex2 = distract(ex, pool, seed)
    return ex2, ex2.extra_facts[0]


# ----------------------------------------------------------------------------- MuSiQue

@dataclass
class MusiqueExample:
    uid: str
    question: str
    answer: str
    aliases: List[str]
    paragraphs: List[dict]        # {'title','paragraph_text','is_supporting'}
    gold_hops: List[Tuple[str, str]]   # (sub-question rendered, sub-answer)
    condition: str = "orig"
    n_dist: int = 4
    order_seed: int = 0

    @property
    def k(self) -> int:
        return len(self.gold_hops)

    def context(self) -> List[dict]:
        sup = [p for p in self.paragraphs if p["is_supporting"]]
        dis = [p for p in self.paragraphs if not p["is_supporting"]]
        rng = random.Random(f"ms-dist-{self.uid}")
        rng.shuffle(dis)
        paras = sup + dis[: self.n_dist]
        rng2 = random.Random(f"ms-order-{self.uid}-{self.order_seed}")
        rng2.shuffle(paras)
        return paras

    def context_text(self) -> str:
        return "\n".join(f"[{i + 1}] {p['title']}: {p['paragraph_text']}"
                         for i, p in enumerate(self.context()))

    def distractor_titles(self) -> List[str]:
        return [p["title"] for p in self.context() if not p["is_supporting"]]

    def answer_prefix(self) -> str:
        return "Answer:"


def _render_subq(q: str, answers: List[str]) -> str:
    for i, a in enumerate(answers):
        q = q.replace(f"#{i + 1}", a)
    if ">>" in q:
        ent, rel = [s.strip() for s in q.split(">>", 1)]
        return f"What is the {rel} of {ent}?"
    q = q.strip()
    return q if q.endswith("?") else q + "?"


def load_musique(split: str = "dev") -> List[MusiqueExample]:
    from huggingface_hub import hf_hub_download
    p = hf_hub_download("bdsaglam/musique", f"musique_ans_v1.0_{split}.jsonl", repo_type="dataset")
    out = []
    with open(p, encoding="utf-8") as f:
        for line in f:
            d = json.loads(line)
            answers = [s["answer"] for s in d["question_decomposition"]]
            hops = [(_render_subq(s["question"], answers[:i]), s["answer"])
                    for i, s in enumerate(d["question_decomposition"])]
            out.append(MusiqueExample(uid=d["id"], question=d["question"], answer=d["answer"],
                                      aliases=d.get("answer_aliases") or [],
                                      paragraphs=d["paragraphs"], gold_hops=hops))
    return out


def musique_condition(ex: MusiqueExample, cond: str) -> MusiqueExample:
    """MuSiQue interventions: number of distractor paragraphs and paragraph order."""
    if cond == "orig":
        return replace(ex, condition="orig", n_dist=4, order_seed=0)
    if cond == "dist0":
        return replace(ex, condition="dist0", n_dist=0)
    if cond == "dist8":
        return replace(ex, condition="dist8", n_dist=8)
    if cond == "order":
        return replace(ex, condition="order", order_seed=1)
    raise ValueError(cond)


In [ ]:
%%writefile hgcd/chain.py
"""Hop-structured chains, the Directed Reasoning Graph (DRG) and do()-interventions.

A chain is a list of hops rendered as lines

    Hop i: <body> => <result>

DRG edge i -> j  (i < j)  iff  result_i is mentioned in body_j.

Interventions on hop j
  do_naive(j)  : mask hop j only. Estimates the *controlled direct effect* of h_j on the
                 answer, holding all downstream hops fixed.
  do_total(j)  : mask hop j and, in every DRG-descendant d of j, mask every mention of
                 result_j and of result_d (descendant values are functions of h_j, so under
                 do(h_j) they are unknown). Deterministic approximation of the *total effect*
                 without re-generating the downstream chain.
  placebo(j)   : content-preserving rewrite of hop j (true effect = 0).
"""

from __future__ import annotations

import re
from dataclasses import dataclass, replace
from typing import Dict, List, Optional, Set

MASK = "[MASKED]"
HOP_RE = re.compile(r"^\s*Hop\s*(\d+)\s*:\s*(.*?)\s*=>\s*(.+?)\s*$")


@dataclass
class Hop:
    idx: int
    body: str
    result: str


def _clean_result(r: str) -> str:
    r = r.strip().strip(".").strip()
    r = re.sub(r"\s*\(.*?\)\s*$", "", r)  # drop trailing "(relation)" annotations
    return r.strip()


def parse_chain(text: str) -> List[Hop]:
    hops = []
    for line in text.splitlines():
        if line.strip().lower().startswith("answer"):
            break
        m = HOP_RE.match(line)
        if m:
            res = _clean_result(m.group(3))
            if res:
                hops.append(Hop(int(m.group(1)), m.group(2).strip(), res))
    return hops


def render_chain(hops: List[Hop]) -> str:
    return "".join(f"Hop {i + 1}: {h.body} => {h.result}\n" for i, h in enumerate(hops))


def _mentions(text: str, ent: str) -> bool:
    if not ent or ent == MASK or len(ent) < 2:
        return False
    return re.search(_ent_pattern(ent), text, flags=re.IGNORECASE) is not None


def _ent_pattern(ent: str) -> str:
    pre = r"\b" if ent[0].isalnum() else ""
    post = r"\b" if ent[-1].isalnum() else ""
    return pre + re.escape(ent) + post


def _mask_ents(text: str, ents: Set[str]) -> str:
    for e in sorted(ents, key=len, reverse=True):
        if e and e != MASK and len(e) >= 2:
            text = re.sub(_ent_pattern(e), MASK, text, flags=re.IGNORECASE)
    return text


# ----------------------------------------------------------------------------- DRG

def drg_edges(hops: List[Hop]) -> List[tuple]:
    return [(i, j) for j in range(len(hops)) for i in range(j)
            if _mentions(hops[j].body, hops[i].result)]


def descendants(hops: List[Hop], j: int) -> List[int]:
    children: Dict[int, List[int]] = {}
    for a, b in drg_edges(hops):
        children.setdefault(a, []).append(b)
    seen, stack = set(), [j]
    while stack:
        for c in children.get(stack.pop(), []):
            if c not in seen:
                seen.add(c)
                stack.append(c)
    return sorted(seen)


def ancestors(hops: List[Hop], j: int) -> List[int]:
    parents: Dict[int, List[int]] = {}
    for a, b in drg_edges(hops):
        parents.setdefault(b, []).append(a)
    seen, stack = set(), [j]
    while stack:
        for p in parents.get(stack.pop(), []):
            if p not in seen:
                seen.add(p)
                stack.append(p)
    return sorted(seen)


def norm(s: str) -> str:
    return re.sub(r"[^a-z0-9 ]", "", s.lower()).strip()


def relevant_hops(hops: List[Hop], target: Optional[str] = None) -> List[int]:
    """Hops with a directed DRG path to the answer (ancestors-or-self of terminal hops).

    Terminal hops: hops whose result is the query target (CLUTRR); else the last hop.
    """
    if not hops:
        return []
    terms = [i for i, h in enumerate(hops) if target and norm(h.result) == norm(target)]
    if not terms:
        terms = [len(hops) - 1]
    rel = set()
    for t in terms:
        rel.add(t)
        rel.update(ancestors(hops, t))
    return sorted(rel)


# ----------------------------------------------------------------------------- do()

def do_naive(hops: List[Hop], j: int) -> List[Hop]:
    out = list(hops)
    out[j] = replace(out[j], body=MASK, result=MASK)
    return out


def do_total(hops: List[Hop], j: int) -> List[Hop]:
    desc = descendants(hops, j)
    ents = {hops[j].result} | {hops[d].result for d in desc}
    out = list(hops)
    out[j] = replace(out[j], body=MASK, result=MASK)
    for d in desc:
        out[d] = replace(out[d], body=_mask_ents(out[d].body, ents),
                         result=_mask_ents(out[d].result, ents))
    return out


CLUTRR_HOP_RE = re.compile(r"^(.+?)'s ([\w-]+) is (.+?)\.?$")


def placebo(hops: List[Hop], j: int) -> List[Hop]:
    """Content-preserving rewrite of hop j (same entities, same relation)."""
    out = list(hops)
    b = hops[j].body
    m = CLUTRR_HOP_RE.match(b)
    if m:
        nb = f"{m.group(3)} is the {m.group(2)} of {m.group(1)}."
    elif b.endswith("?"):
        nb = "To restate: " + b
    else:
        nb = "In other words, " + b
    out[j] = replace(out[j], body=nb)
    return out


def mask_marker(hops: List[Hop], j: int) -> List[Hop]:
    """Artifact control: insert the mask token into hop j WITHOUT removing any content.
    Its effect isolates the out-of-distribution cost of the [MASKED] token itself."""
    out = list(hops)
    out[j] = replace(out[j], body=f"{MASK} {hops[j].body}")
    return out


In [ ]:
%%writefile hgcd/prompts.py
"""Few-shot prompts that elicit hop-structured chains."""

from __future__ import annotations

from typing import List

from .chain import Hop, render_chain

CLUTRR_SYSTEM = (
    "You solve family-relation puzzles. Walk from the first person to the second person "
    "one relation at a time, using only facts stated in the story. Write one line per hop "
    "in the form 'Hop i: X's <relation> is Y. => Y', then give the final answer.")

CLUTRR_SHOTS = [
    {
        "story": "Mabel took her son Rupert to the zoo. Rupert and his sister Agatha baked a cake "
                 "for their grandmother.",
        "src": "Mabel", "tgt": "Agatha",
        "hops": [("Mabel's son is Rupert.", "Rupert"), ("Rupert's sister is Agatha.", "Agatha")],
        "ans": "daughter",
    },
    {
        "story": "Clement went fishing with his father Bertram. Bertram called his sister Winifred "
                 "to wish her a happy birthday. Winifred was proud of her daughter Odette.",
        "src": "Clement", "tgt": "Odette",
        "hops": [("Clement's father is Bertram.", "Bertram"),
                 ("Bertram's sister is Winifred.", "Winifred"),
                 ("Winifred's daughter is Odette.", "Odette")],
        "ans": "cousin",
    },
]
# The second shot uses a relation outside the CLUTRR label set on purpose: it teaches the
# format without leaking label statistics. Its answer is never a scoring candidate.

MUSIQUE_SYSTEM = (
    "You answer multi-hop questions using the given paragraphs. Break the question into "
    "single-hop sub-questions, answer each in one line of the form "
    "'Hop i: <sub-question> => <answer>', then give the final answer.")

MUSIQUE_SHOTS = [
    {
        "context": "[1] Thornfield Abbey: Thornfield Abbey is a novel written by Edwina Marsh.\n"
                   "[2] Edwina Marsh: Edwina Marsh was born in Galway in 1902.\n"
                   "[3] Galway Bay: Galway Bay is a large bay on the west coast of Ireland.",
        "question": "Where was the author of Thornfield Abbey born?",
        "hops": [("Who is the author of Thornfield Abbey?", "Edwina Marsh"),
                 ("Where was Edwina Marsh born?", "Galway")],
        "ans": "Galway",
    },
]


def _clutrr_user(story: str, src: str, tgt: str) -> str:
    return f"Story: {story}\nQuestion: How is {tgt} related to {src}?"


def _hops_text(pairs) -> str:
    return render_chain([Hop(i + 1, b, r) for i, (b, r) in enumerate(pairs)])


def clutrr_messages(story: str, src: str, tgt: str) -> List[dict]:
    msgs = [{"role": "system", "content": CLUTRR_SYSTEM}]
    for s in CLUTRR_SHOTS:
        msgs.append({"role": "user", "content": _clutrr_user(s["story"], s["src"], s["tgt"])})
        msgs.append({"role": "assistant", "content": _hops_text(s["hops"]) +
                     f"Answer: {s['tgt']} is {s['src']}'s {s['ans']}."})
    msgs.append({"role": "user", "content": _clutrr_user(story, src, tgt)})
    return msgs


def musique_messages(context: str, question: str) -> List[dict]:
    msgs = [{"role": "system", "content": MUSIQUE_SYSTEM}]
    for s in MUSIQUE_SHOTS:
        msgs.append({"role": "user", "content": f"Paragraphs:\n{s['context']}\nQuestion: {s['question']}"})
        msgs.append({"role": "assistant", "content": _hops_text(s["hops"]) + f"Answer: {s['ans']}"})
    msgs.append({"role": "user", "content": f"Paragraphs:\n{context}\nQuestion: {question}"})
    return msgs


def musique_question_only(question: str) -> List[dict]:
    """No-context prompt (for CAD's contrast distribution)."""
    return musique_messages("(no paragraphs given)", question)


def clutrr_question_only(src: str, tgt: str) -> List[dict]:
    return clutrr_messages("(no story given)", src, tgt)


In [ ]:
%%writefile hgcd/lm.py
"""Thin HF-transformers wrapper: chat prompts, batched generation, KV-cached
candidate scoring, and contrastive (two-context) greedy decoding."""

from __future__ import annotations

import math
from typing import List, Optional, Sequence

import numpy as np
import torch
import torch.nn.functional as F


class LM:
    def __init__(self, name: str, device: Optional[str] = None, dtype=None,
                 device_map: Optional[str] = None):
        from transformers import AutoModelForCausalLM, AutoTokenizer
        self.name = name
        self.tok = AutoTokenizer.from_pretrained(name)
        self.tok.padding_side = "left"
        if self.tok.pad_token is None:
            self.tok.pad_token = self.tok.eos_token
        cuda = torch.cuda.is_available()
        dtype = dtype or (torch.float16 if cuda else torch.float32)
        kw = dict(device_map=device_map) if device_map else {}
        try:
            self.model = AutoModelForCausalLM.from_pretrained(name, dtype=dtype, **kw)
        except TypeError:
            self.model = AutoModelForCausalLM.from_pretrained(name, torch_dtype=dtype, **kw)
        if not device_map:
            self.model.to(device or ("cuda" if cuda else "cpu"))
        self.model.eval()
        self.device = next(self.model.parameters()).device
        self.n_boundary_fallback = 0

    # ------------------------------------------------------------------ prompts
    def prompt(self, messages: List[dict], assistant_prefix: str = "") -> str:
        p = self.tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        return p + assistant_prefix

    def _enc(self, texts: Sequence[str]):
        return self.tok(list(texts), return_tensors="pt", padding=True,
                        add_special_tokens=False).to(self.device)

    # ------------------------------------------------------------------ generation
    @torch.no_grad()
    def generate(self, prompts: List[str], max_new_tokens: int = 160, do_sample: bool = False,
                 temperature: float = 0.7, n: int = 1, batch_size: int = 8,
                 stop: str = "Answer:") -> List[str]:
        outs = []
        for i in range(0, len(prompts), batch_size):
            enc = self._enc(prompts[i:i + batch_size])
            kw = dict(max_new_tokens=max_new_tokens, do_sample=do_sample,
                      num_return_sequences=n, pad_token_id=self.tok.pad_token_id)
            if do_sample:
                kw.update(temperature=temperature, top_p=1.0, top_k=0)
            else:
                kw.update(temperature=None, top_p=None, top_k=None)
            try:
                g = self.model.generate(**enc, stop_strings=[stop], tokenizer=self.tok, **kw)
            except (TypeError, ValueError):
                g = self.model.generate(**enc, **kw)
            new = g[:, enc.input_ids.shape[1]:]
            for t in self.tok.batch_decode(new, skip_special_tokens=True):
                outs.append(t.split(stop)[0])
        return outs

    # ------------------------------------------------------------------ scoring
    def _cand_ids(self, p_ids: List[int], prefix: str, cands: Sequence[str]) -> List[List[int]]:
        out = []
        for c in cands:
            full = self.tok(prefix + c, add_special_tokens=False).input_ids
            if full[:len(p_ids)] == p_ids and len(full) > len(p_ids):
                out.append(full[len(p_ids):])
            else:  # BPE merge across the boundary: score the candidate tokenized alone
                self.n_boundary_fallback += 1
                out.append(self.tok(c, add_special_tokens=False).input_ids or [self.tok.eos_token_id])
        return out

    @torch.no_grad()
    def score(self, prefix: str, cands: Sequence[str]) -> np.ndarray:
        """Sum of log-probs of each candidate continuation after `prefix` (KV-cached)."""
        p_ids = self.tok(prefix, add_special_tokens=False).input_ids
        cids = self._cand_ids(p_ids, prefix, cands)
        inp = torch.tensor([p_ids], device=self.device)
        out = self.model(input_ids=inp, use_cache=True)
        first = F.log_softmax(out.logits[0, -1].float(), -1)
        res = np.array([first[c[0]].item() for c in cids], dtype=np.float64)
        L = max(len(c) for c in cids)
        if L == 1:
            return res
        C, P = len(cids), len(p_ids)
        x = torch.full((C, L - 1), self.tok.pad_token_id, dtype=torch.long, device=self.device)
        att = torch.zeros((C, P + L - 1), dtype=torch.long, device=self.device)
        att[:, :P] = 1
        for i, c in enumerate(cids):
            if len(c) > 1:
                x[i, :len(c) - 1] = torch.tensor(c[:-1], device=self.device)
                att[i, P:P + len(c) - 1] = 1
        pos = torch.arange(P, P + L - 1, device=self.device).unsqueeze(0).expand(C, -1)
        try:
            past = out.past_key_values
            past.batch_repeat_interleave(C)
            o2 = self.model(input_ids=x, attention_mask=att, position_ids=pos,
                            past_key_values=past, use_cache=False)
            lp = F.log_softmax(o2.logits.float(), -1)
        except Exception:  # cache API differs across versions: recompute without cache
            full = torch.cat([inp.expand(C, -1), x], 1)
            o2 = self.model(input_ids=full, attention_mask=att)
            lp = F.log_softmax(o2.logits[:, P:].float(), -1)
        for i, c in enumerate(cids):
            for t in range(1, len(c)):
                res[i] += lp[i, t - 1, c[t]].item()
        return res

    # ------------------------------------------------------------------ L1 contrastive decoding
    @torch.no_grad()
    def generate_contrastive(self, real: List[str], cf: List[str], alpha: float = 1.0,
                             beta: float = 0.1, max_new_tokens: int = 160, gate: str = "result",
                             stop: str = "Answer:", batch_size: int = 4,
                             space: str = "prob") -> List[str]:
        """Greedy decoding with score (1+a)*p(.|real) - a*p(.|cf)  (space='prob', default)
        or (1+a)*log p(.|real) - a*log p(.|cf)  (space='log', CAD-style).

        gate='result': contrast only while emitting a hop result (text after '=>' on the
        current line); gate='all': contrast every token (CAD-style).
        Plausibility constraint: only tokens with p_real >= beta * max p_real are eligible.
        """
        outs = []
        for i in range(0, len(real), batch_size):
            outs.extend(self._contrastive_batch(real[i:i + batch_size], cf[i:i + batch_size],
                                                alpha, beta, max_new_tokens, gate, stop, space))
        return outs

    def _contrastive_batch(self, real, cf, alpha, beta, max_new_tokens, gate, stop, space):
        B = len(real)
        enc = self._enc(list(real) + list(cf))
        att = enc.attention_mask
        pos = (att.cumsum(-1) - 1).clamp(min=0)
        out = self.model(input_ids=enc.input_ids, attention_mask=att, position_ids=pos,
                         use_cache=True)
        past, logits = out.past_key_values, out.logits[:, -1]
        last_pos = pos[:, -1]
        gen = [[] for _ in range(B)]
        text = [""] * B
        done = [False] * B
        log_beta = math.log(beta)
        eos = {self.tok.eos_token_id, self.tok.pad_token_id}
        for _ in range(max_new_tokens):
            lp = F.log_softmax(logits.float(), -1)
            nxt = []
            for b in range(B):
                if done[b]:
                    nxt.append(self.tok.pad_token_id)
                    continue
                lr, lc = lp[b], lp[B + b]
                cur_line = text[b].rsplit("\n", 1)[-1]
                on = gate == "all" or (gate == "result" and "=>" in cur_line)
                if on:
                    if space == "prob":
                        s = (1 + alpha) * lr.exp() - alpha * lc.exp()
                    else:
                        s = (1 + alpha) * lr - alpha * lc
                    s[lr < lr.max() + log_beta] = -float("inf")
                    t = int(s.argmax())
                else:
                    t = int(lr.argmax())
                gen[b].append(t)
                text[b] = self.tok.decode(gen[b], skip_special_tokens=True)
                if t in eos or stop in text[b]:
                    done[b] = True
                nxt.append(t)
            if all(done):
                break
            new = torch.tensor(nxt + nxt, device=self.device).unsqueeze(1)
            att = torch.cat([att, torch.ones((2 * B, 1), dtype=att.dtype, device=self.device)], 1)
            last_pos = last_pos + 1
            out = self.model(input_ids=new, attention_mask=att, position_ids=last_pos.unsqueeze(1),
                             past_key_values=past, use_cache=True)
            past, logits = out.past_key_values, out.logits[:, -1]
        return [t.split(stop)[0] for t in text]


In [ ]:
%%writefile hgcd/decoding.py
"""Answer-level decoding rules over a finite candidate set.

All rules consume *normalized* candidate distributions computed once per example, so any
alpha can be evaluated post hoc (cheap sweeps, no extra forward passes).

Notation (per example):
  p_cot      p(a | Q, P, y)              model's own chain y
  p_dir      p(a | Q, P)                 direct answer, no chain
  p_noctx    p(a | Q)                    no context, no chain
  p_tot[j]   p(a | Q, P, do_total(y, j))
  p_nai[j]   p(a | Q, P, do_naive(y, j))
  p_pla[j]   p(a | Q, P, placebo(y, j))
  rel        indices of DRG-relevant hops (directed path to the answer)
"""

from __future__ import annotations

from typing import Dict, List, Optional

import numpy as np

EPS = 1e-12


def softmax_logp(lp: np.ndarray) -> np.ndarray:
    z = lp - lp.max()
    e = np.exp(z)
    return e / e.sum()


def jsd(p: np.ndarray, q: np.ndarray) -> float:
    """Jensen-Shannon divergence in bits, in [0, 1]."""
    m = 0.5 * (p + q)

    def kl(a, b):
        return float(np.sum(a * (np.log2(a + EPS) - np.log2(b + EPS))))
    return max(0.0, min(1.0, 0.5 * kl(p, m) + 0.5 * kl(q, m)))


def _plausible(p: np.ndarray, beta: float) -> np.ndarray:
    return p >= beta * p.max()


def _argmax_masked(s: np.ndarray, mask: np.ndarray) -> int:
    s = np.where(mask, s, -np.inf)
    return int(np.argmax(s))


def log_contrast(p_pos: np.ndarray, p_neg: np.ndarray, alpha: float, beta: float) -> int:
    s = (1 + alpha) * np.log(p_pos + EPS) - alpha * np.log(p_neg + EPS)
    return _argmax_masked(s, _plausible(p_pos, beta))


def prob_contrast(p_pos: np.ndarray, p_neg: np.ndarray, alpha: float, beta: float) -> int:
    """Probability-space contrast (1+a) p - a p_neg (Prop. 1: cancels shortcut mixtures)."""
    s = (1 + alpha) * p_pos - alpha * p_neg
    return _argmax_masked(s, _plausible(p_pos, beta))


def _mix(ps: List[np.ndarray], idx: List[int]) -> Optional[np.ndarray]:
    if not idx:
        return None
    return np.mean([ps[j] for j in idx], axis=0)


METHODS = ["direct", "cot", "cad", "adacad", "cad_chain", "hgcd", "hgcd_log", "hgcd_naive",
           "hgcd_uniform", "hgcd_placebo", "hgcd_ada"]


def decide(d: Dict, method: str, alpha: float = 1.0, beta: float = 0.1) -> int:
    """Return the index of the chosen candidate for one example record `d`."""
    p_cot, p_dir, p_noctx = d["p_cot"], d["p_dir"], d["p_noctx"]
    n_hops = len(d["p_tot"])
    rel = d["rel"]
    if method == "direct":
        return int(np.argmax(p_dir))
    if method == "cot":
        return int(np.argmax(p_cot))
    if method == "sc":  # self-consistency majority vote; ties broken by p_cot
        votes = np.bincount(d["sc_votes"], minlength=len(p_cot)).astype(float)
        return int(np.argmax(votes + 1e-3 * p_cot))
    if method == "cad":
        return log_contrast(p_dir, p_noctx, alpha, beta)
    if method == "adacad":
        return log_contrast(p_dir, p_noctx, jsd(p_dir, p_noctx), beta)
    if method == "cad_chain":
        return log_contrast(p_cot, p_dir, alpha, beta)
    # ---- HGCD family: contrast the chain against do()-interventions on its own hops
    if n_hops == 0:
        return int(np.argmax(p_cot))
    if method == "hgcd":
        neg = _mix(d["p_tot"], rel)
    elif method == "hgcd_log":
        neg = _mix(d["p_tot"], rel)
        return int(np.argmax(p_cot)) if neg is None else log_contrast(p_cot, neg, alpha, beta)
    elif method == "hgcd_naive":
        neg = _mix(d["p_nai"], rel)
    elif method == "hgcd_uniform":
        neg = _mix(d["p_tot"], list(range(n_hops)))
    elif method == "hgcd_placebo":
        neg = _mix(d["p_pla"], rel)
    elif method == "hgcd_ada":
        neg = _mix(d["p_tot"], rel)
        if neg is None:
            return int(np.argmax(p_cot))
        return prob_contrast(p_cot, neg, alpha * jsd(p_cot, neg), beta)
    else:
        raise ValueError(method)
    if neg is None:
        return int(np.argmax(p_cot))
    return prob_contrast(p_cot, neg, alpha, beta)


In [ ]:
%%writefile hgcd/experiments.py
"""Experiment runners. Each runner returns plain-dict records that store the candidate
distributions, so every decoding rule / alpha is evaluated post hoc."""

from __future__ import annotations

import random
import time
from typing import Dict, List, Optional

import numpy as np

from . import chain as C
from . import data as D
from .decoding import softmax_logp
from .prompts import (clutrr_messages, clutrr_question_only, musique_messages,
                      musique_question_only)


def _log(msg: str):
    print(time.strftime("[%H:%M:%S] "), msg, flush=True)


def _dist(lm, prefix: str, cands: List[str]) -> np.ndarray:
    return softmax_logp(lm.score(prefix, cands))


def _score_chain_contexts(lm, base_prompt: str, hops: List[C.Hop], ans_prefix: str,
                          cands: List[str], full: bool, marker: bool = False) -> Dict:
    """p(a | context, intervened chain) for every hop and intervention type."""
    def ctx(hs):
        return base_prompt + C.render_chain(hs) + ans_prefix
    rec = {"p_cot": _dist(lm, ctx(hops), cands),
           "p_tot": [_dist(lm, ctx(C.do_total(hops, j)), cands) for j in range(len(hops))]}
    if full:
        rec["p_nai"] = [_dist(lm, ctx(C.do_naive(hops, j)), cands) for j in range(len(hops))]
        rec["p_pla"] = [_dist(lm, ctx(C.placebo(hops, j)), cands) for j in range(len(hops))]
        if marker:
            rec["p_mm"] = [_dist(lm, ctx(C.mask_marker(hops, j)), cands) for j in range(len(hops))]
    else:
        rec["p_nai"] = rec["p_pla"] = None
    return rec


# ============================================================================ CLUTRR

def clutrr_score(lm, ex: D.ClutrrExample, chain_text: str, rels: List[str],
                 full: bool = True) -> Dict:
    cands = [" " + r for r in rels]
    hops = C.parse_chain(chain_text)
    base = lm.prompt(clutrr_messages(ex.story_text, ex.source, ex.target))
    rec = _score_chain_contexts(lm, base, hops, ex.answer_prefix(), cands, full)
    rec["p_dir"] = _dist(lm, base + ex.answer_prefix(), cands)
    rec["p_noctx"] = _dist(lm, lm.prompt(clutrr_question_only(ex.source, ex.target))
                           + ex.answer_prefix(), cands)
    rec["rel"] = C.relevant_hops(hops, target=ex.target)
    rec["drg_connected"] = bool(hops) and len(rec["rel"]) == len(hops)
    gold_tails = [h.tail for h in ex.gold_hops]
    res = [C.norm(h.result) for h in hops]
    rec.update(uid=ex.uid, k=ex.k, cond=ex.condition, gold=rels.index(ex.answer),
               n_hops=len(hops), fmt_ok=len(hops) > 0, chain=chain_text,
               hop_fid=float(np.mean([C.norm(t) in res for t in gold_tails])),
               cands=rels)
    return rec


def _gen_by_k(lm, prompts: List[str], ks: List[int], batch_size: int) -> List[str]:
    out = [None] * len(prompts)
    for k in sorted(set(ks)):
        idx = [i for i, kk in enumerate(ks) if kk == k]
        gens = lm.generate([prompts[i] for i in idx], max_new_tokens=26 * (k + 2) + 16,
                           batch_size=batch_size)
        for i, g in zip(idx, gens):
            out[i] = g
    return out


def run_clutrr(lm, examples: List[D.ClutrrExample], pool: List[D.ClutrrExample],
               rels: List[str], conditions=("orig", "rename", "order", "distract", "placebo_input"),
               batch_size: int = 8, l1: bool = False, l1_alpha: float = 1.0,
               l1_gate: str = "result", sc_n: int = 0) -> List[Dict]:
    """sc_n > 0 adds a self-consistency baseline (sc_n sampled chains, T=0.7, majority vote)
    on the 'orig' condition. It costs sc_n extra generations per question, i.e. strictly more
    compute than HGCD, which needs no extra generation."""
    recs = []
    for cond in conditions:
        exs = [D.apply_condition(e, cond, pool) for e in examples]
        prompts = [lm.prompt(clutrr_messages(e.story_text, e.source, e.target)) for e in exs]
        _log(f"CLUTRR [{cond}] generating {len(exs)} chains")
        gens = _gen_by_k(lm, prompts, [e.k for e in exs], batch_size)
        _log(f"CLUTRR [{cond}] scoring")
        votes = [None] * len(exs)
        if sc_n > 0 and cond == "orig":
            _log(f"CLUTRR [sc] sampling {sc_n} chains per question")
            cands = [" " + r for r in rels]
            for k in sorted(set(e.k for e in exs)):
                idx = [i for i, e in enumerate(exs) if e.k == k]
                samples = lm.generate([prompts[i] for i in idx], max_new_tokens=26 * (k + 2) + 16,
                                      do_sample=True, n=sc_n, batch_size=max(1, batch_size // 2))
                for t, i in enumerate(idx):
                    vs = []
                    for c in samples[t * sc_n:(t + 1) * sc_n]:
                        pre = prompts[i] + C.render_chain(C.parse_chain(c)) + exs[i].answer_prefix()
                        vs.append(int(np.argmax(lm.score(pre, cands))))
                    votes[i] = vs
        for e, g, v in zip(exs, gens, votes):
            r = clutrr_score(lm, e, g, rels, full=(cond == "orig"))
            r["chain_src"] = "greedy"
            r["sc_votes"] = v
            recs.append(r)
        if l1 and cond == "orig":
            _log(f"CLUTRR [L1] contrastive generation ({l1_gate}, a={l1_alpha})")
            cf = [lm.prompt(clutrr_messages(D.counterfactual_story(e), e.source, e.target))
                  for e in exs]
            order = sorted(range(len(exs)), key=lambda i: exs[i].k)
            for s in range(0, len(order), batch_size):
                idx = order[s:s + batch_size]
                kmax = max(exs[i].k for i in idx)
                gl1 = lm.generate_contrastive([prompts[i] for i in idx], [cf[i] for i in idx],
                                              alpha=l1_alpha, gate=l1_gate,
                                              max_new_tokens=26 * (kmax + 2) + 16,
                                              batch_size=batch_size)
                for i, g in zip(idx, gl1):
                    r = clutrr_score(lm, exs[i], g, rels, full=False)
                    r["chain_src"] = "l1"
                    recs.append(r)
    return recs


# ============================================================================ estimator validation

def _corrupt_clutrr_hop(h: C.Hop, rels: List[str], rng: random.Random) -> C.Hop:
    m = C.CLUTRR_HOP_RE.match(h.body)
    if not m:
        return C.Hop(h.idx, C.MASK, h.result)
    alt = rng.choice([r for r in rels if r != m.group(2)])
    return C.Hop(h.idx, f"{m.group(1)}'s {alt} is {m.group(3)}.", h.result)


def run_estimators_clutrr(lm, examples: List[D.ClutrrExample], pool: List[D.ClutrrExample],
                          rels: List[str], rollouts: int = 0, batch_size: int = 8,
                          seed: int = 0) -> List[Dict]:
    """Gold chains + one inserted true-but-redundant hop (true necessity 0)."""
    cands = [" " + r for r in rels]
    rows = []
    for n, ex in enumerate(examples):
        rng = random.Random(f"est-{ex.uid}-{seed}")
        ex2, red = D.with_redundant_fact(ex, pool, seed)
        hops = [C.Hop(i + 1, h.text(), h.tail) for i, h in enumerate(ex.gold_hops)]
        rpos = rng.randint(0, len(hops) - 1)
        hops.insert(rpos, C.Hop(0, red.text(), red.tail))
        base = lm.prompt(clutrr_messages(ex2.story_text, ex2.source, ex2.target))
        pre = ex2.answer_prefix()
        g = rels.index(ex.answer)
        recs = _score_chain_contexts(lm, base, hops, pre, cands, full=True, marker=True)
        p_full = recs["p_cot"]
        rel = C.relevant_hops(hops, target=ex.target)
        pns = [np.nan] * len(hops)
        if rollouts > 0:
            parts = [hops[:j] + [_corrupt_clutrr_hop(hops[j], rels, rng)] for j in range(len(hops))]
            prompts = [base + C.render_chain(part) for part in parts]
            conts = lm.generate(prompts, max_new_tokens=26 * (len(hops) + 1) + 16,
                                do_sample=True, n=rollouts, batch_size=max(1, batch_size // 2))
            y_full = float(np.argmax(p_full) == g)
            for j in range(len(hops)):
                ys = []
                for c in conts[j * rollouts:(j + 1) * rollouts]:
                    allh = parts[j] + C.parse_chain(c)
                    p = _dist(lm, base + C.render_chain(allh) + pre, cands)
                    ys.append(float(np.argmax(p) == g))
                pns[j] = y_full - float(np.mean(ys))
        for j in range(len(hops)):
            rows.append(dict(
                dataset="clutrr", uid=ex.uid, k=ex.k, hop=j, n_hops=len(hops),
                label=int(j != rpos), is_final=int(j == len(hops) - 1),
                drg_relevant=int(j in rel), p_full_gold=float(p_full[g]),
                full_correct=int(np.argmax(p_full) == g),
                d_total=float(p_full[g] - recs["p_tot"][j][g]),
                d_naive=float(p_full[g] - recs["p_nai"][j][g]),
                d_placebo=float(p_full[g] - recs["p_pla"][j][g]),
                d_maskmark=float(p_full[g] - recs["p_mm"][j][g]),
                pns_rollout=pns[j]))
        if (n + 1) % 10 == 0:
            _log(f"estimators clutrr {n + 1}/{len(examples)}")
    return rows


def _musique_cands(ex: D.MusiqueExample, extra: List[str]) -> List[str]:
    seen, out = set(), []
    for c in extra + [ex.answer] + [h[1] for h in ex.gold_hops] + \
            [p["title"] for p in ex.context()]:
        k = C.norm(c)
        if k and k not in seen:
            seen.add(k)
            out.append(c)
    return out


def run_estimators_musique(lm, examples: List[D.MusiqueExample], seed: int = 0) -> List[Dict]:
    """MuSiQue gold chains (bridge-entity graphs) + an inserted redundant hop."""
    rows = []
    rng = random.Random(seed)
    for n, ex in enumerate(examples):
        other = rng.choice([o for o in examples if o.uid != ex.uid])
        hops = [C.Hop(i + 1, q, a) for i, (q, a) in enumerate(ex.gold_hops)]
        rpos = rng.randint(0, len(hops) - 1)
        hops.insert(rpos, C.Hop(0, other.gold_hops[0][0], other.gold_hops[0][1]))
        cands = [" " + c for c in _musique_cands(ex, [])]
        base = lm.prompt(musique_messages(ex.context_text(), ex.question))
        recs = _score_chain_contexts(lm, base, hops, ex.answer_prefix(), cands, full=True,
                                     marker=True)
        g = [C.norm(c) for c in cands].index(C.norm(ex.answer))
        p_full = recs["p_cot"]
        rel = C.relevant_hops(hops)
        for j in range(len(hops)):
            rows.append(dict(
                dataset="musique", uid=ex.uid, k=ex.k, hop=j, n_hops=len(hops),
                label=int(j != rpos), is_final=int(j == len(hops) - 1),
                drg_relevant=int(j in rel), p_full_gold=float(p_full[g]),
                full_correct=int(np.argmax(p_full) == g),
                d_total=float(p_full[g] - recs["p_tot"][j][g]),
                d_naive=float(p_full[g] - recs["p_nai"][j][g]),
                d_placebo=float(p_full[g] - recs["p_pla"][j][g]),
                d_maskmark=float(p_full[g] - recs["p_mm"][j][g]),
                pns_rollout=np.nan))
        if (n + 1) % 10 == 0:
            _log(f"estimators musique {n + 1}/{len(examples)}")
    return rows


# ============================================================================ MuSiQue decoding

def run_musique(lm, examples: List[D.MusiqueExample], conditions=("orig", "dist8", "order"),
                batch_size: int = 4) -> List[Dict]:
    recs = []
    for cond in conditions:
        exs = [D.musique_condition(e, cond) for e in examples]
        prompts = [lm.prompt(musique_messages(e.context_text(), e.question)) for e in exs]
        _log(f"MuSiQue [{cond}] generating {len(exs)} chains")
        gens = lm.generate(prompts, max_new_tokens=200, batch_size=batch_size)
        p2 = [p + C.render_chain(C.parse_chain(g)) + "Answer:" for p, g in zip(prompts, gens)]
        free = lm.generate(p2, max_new_tokens=24, batch_size=batch_size, stop="\n")
        _log(f"MuSiQue [{cond}] scoring")
        for e, p, g, fa in zip(exs, prompts, gens, free):
            hops = C.parse_chain(g)
            fa = fa.strip().strip(".")
            cand_txt = _musique_cands_model(e, [fa] + [h.result for h in hops])
            cands = [" " + c for c in cand_txt]
            r = _score_chain_contexts(lm, p, hops, e.answer_prefix(), cands, full=(cond == "orig"))
            r["sc_votes"] = None
            r["p_dir"] = _dist(lm, p + e.answer_prefix(), cands)
            r["p_noctx"] = _dist(lm, lm.prompt(musique_question_only(e.question))
                                 + e.answer_prefix(), cands)
            r["rel"] = C.relevant_hops(hops)
            r.update(uid=e.uid, k=e.k, cond=cond, cands=cand_txt, gold_answer=e.answer,
                     aliases=e.aliases, free_answer=fa, n_hops=len(hops),
                     fmt_ok=len(hops) > 0, chain=g, chain_src="greedy")
            recs.append(r)
    return recs


def _musique_cands_model(ex: D.MusiqueExample, model_strings: List[str]) -> List[str]:
    """Candidates without access to gold: model answers, hop results, paragraph titles."""
    seen, out = set(), []
    for c in model_strings + [p["title"] for p in ex.context()]:
        k = C.norm(c)
        if k and k not in seen and c != C.MASK:
            seen.add(k)
            out.append(c)
    return out[:16]


In [ ]:
%%writefile hgcd/metrics.py
"""Evaluation: accuracy, EM/F1, paired effects with bootstrap CIs, McNemar, AUC."""

from __future__ import annotations

import math
import re
import string
from collections import Counter
from typing import Dict, Iterable, List, Optional, Tuple

import numpy as np
import pandas as pd

from .decoding import decide

GREEDY_METHODS = ["direct", "cot", "sc", "cad", "adacad", "cad_chain", "hgcd", "hgcd_log",
                  "hgcd_naive", "hgcd_uniform", "hgcd_placebo", "hgcd_ada"]
NEEDS_FULL = {"hgcd_naive", "hgcd_placebo"}


# ----------------------------------------------------------------------------- answers

def _norm_ans(s: str) -> str:
    s = s.lower()
    s = "".join(ch for ch in s if ch not in set(string.punctuation))
    s = re.sub(r"\b(a|an|the)\b", " ", s)
    return " ".join(s.split())


def em(pred: str, golds: Iterable[str]) -> float:
    return float(any(_norm_ans(pred) == _norm_ans(g) for g in golds))


def f1(pred: str, golds: Iterable[str]) -> float:
    best = 0.0
    pt = _norm_ans(pred).split()
    for g in golds:
        gt = _norm_ans(g).split()
        common = Counter(pt) & Counter(gt)
        ns = sum(common.values())
        if ns == 0:
            continue
        p, r = ns / len(pt), ns / len(gt)
        best = max(best, 2 * p * r / (p + r))
    return best


# ----------------------------------------------------------------------------- statistics

def boot_ci(x: np.ndarray, n: int = 2000, seed: int = 0) -> Tuple[float, float, float]:
    x = np.asarray(x, dtype=float)
    if len(x) == 0:
        return (np.nan, np.nan, np.nan)
    rng = np.random.default_rng(seed)
    bs = rng.choice(x, size=(n, len(x)), replace=True).mean(1)
    return float(x.mean()), float(np.percentile(bs, 2.5)), float(np.percentile(bs, 97.5))


def mcnemar_p(a: np.ndarray, b: np.ndarray) -> float:
    """Exact two-sided McNemar test on paired binary outcomes."""
    a, b = np.asarray(a).astype(bool), np.asarray(b).astype(bool)
    n01, n10 = int(np.sum(~a & b)), int(np.sum(a & ~b))
    n = n01 + n10
    if n == 0:
        return 1.0
    k = min(n01, n10)
    p = sum(math.comb(n, i) for i in range(k + 1)) / 2 ** n
    return min(1.0, 2 * p)


def auc(scores: np.ndarray, labels: np.ndarray) -> float:
    from sklearn.metrics import roc_auc_score
    s, y = np.asarray(scores, float), np.asarray(labels, int)
    ok = ~np.isnan(s)
    if len(set(y[ok])) < 2:
        return np.nan
    return float(roc_auc_score(y[ok], s[ok]))


# ----------------------------------------------------------------------------- decision tables

def clutrr_table(recs: List[Dict], alpha: float = 1.0, beta: float = 0.1,
                 methods: Optional[List[str]] = None) -> pd.DataFrame:
    methods = methods or GREEDY_METHODS
    rows = []
    for r in recs:
        for m in methods:
            if r["chain_src"] == "l1" and m not in ("cot", "hgcd"):
                continue
            if m in NEEDS_FULL and r.get("p_nai") is None:
                continue
            if m == "sc" and not r.get("sc_votes"):
                continue
            i = decide(r, m, alpha, beta)
            name = m if r["chain_src"] == "greedy" else f"l1+{m}"
            rows.append(dict(uid=r["uid"], k=r["k"], cond=r["cond"], method=name,
                             correct=int(i == r["gold"]), fmt_ok=int(r["fmt_ok"]),
                             n_hops=r["n_hops"], hop_fid=r.get("hop_fid", np.nan),
                             drg_connected=int(r.get("drg_connected", False))))
    return pd.DataFrame(rows)


def musique_table(recs: List[Dict], alpha: float = 1.0, beta: float = 0.1,
                  methods: Optional[List[str]] = None) -> pd.DataFrame:
    methods = methods or GREEDY_METHODS
    rows = []
    for r in recs:
        golds = [r["gold_answer"]] + list(r["aliases"])
        rows.append(dict(uid=r["uid"], k=r["k"], cond=r["cond"], method="free",
                         em=em(r["free_answer"], golds), f1=f1(r["free_answer"], golds),
                         cand_recall=max(em(c, golds) for c in r["cands"])))
        for m in methods:
            if m in NEEDS_FULL and r.get("p_nai") is None:
                continue
            if m == "sc" and not r.get("sc_votes"):
                continue
            pred = r["cands"][decide(r, m, alpha, beta)]
            rows.append(dict(uid=r["uid"], k=r["k"], cond=r["cond"], method=m,
                             em=em(pred, golds), f1=f1(pred, golds),
                             cand_recall=max(em(c, golds) for c in r["cands"])))
    df = pd.DataFrame(rows)
    df["correct"] = df["em"]
    return df


def accuracy_summary(df: pd.DataFrame, by: List[str], col: str = "correct") -> pd.DataFrame:
    out = []
    for key, g in df.groupby(by):
        m, lo, hi = boot_ci(g[col].values)
        key = key if isinstance(key, tuple) else (key,)
        out.append(dict(zip(by, key), mean=m, lo=lo, hi=hi, n=len(g)))
    return pd.DataFrame(out)


def ate_table(df: pd.DataFrame, base_cond: str = "orig", col: str = "correct") -> pd.DataFrame:
    """Paired ATE of each binary intervention per method: E[Y(1) - Y(0)] (truth = 0)."""
    out = []
    base = df[df.cond == base_cond].set_index(["uid", "method"])[col]
    for cond in sorted(set(df.cond) - {base_cond}):
        cur = df[df.cond == cond].set_index(["uid", "method"])[col]
        j = pd.concat([base.rename("y0"), cur.rename("y1")], axis=1, join="inner").reset_index()
        for m, g in j.groupby("method"):
            d = (g.y1 - g.y0).values
            mean, lo, hi = boot_ci(d)
            out.append(dict(cond=cond, method=m, ate=mean, lo=lo, hi=hi, abs_ate=abs(mean),
                            n=len(g)))
    return pd.DataFrame(out)


def paired_compare(df: pd.DataFrame, a: str, b: str, cond: str = "orig",
                   col: str = "correct") -> Dict:
    x = df[(df.cond == cond) & (df.method == a)].set_index("uid")[col]
    y = df[(df.cond == cond) & (df.method == b)].set_index("uid")[col]
    j = pd.concat([x.rename("a"), y.rename("b")], axis=1, join="inner")
    mean, lo, hi = boot_ci((j.a - j.b).values)
    return dict(a=a, b=b, diff=mean, lo=lo, hi=hi, p_mcnemar=mcnemar_p(j.a.values > 0.5,
                                                                         j.b.values > 0.5),
                n=len(j))


def estimator_table(rows: List[Dict]) -> Tuple[pd.DataFrame, pd.DataFrame]:
    df = pd.DataFrame(rows)
    aucs = []
    for ds, g in df.groupby("dataset"):
        g2 = g[g.full_correct == 1]  # PS = 1 condition (as in PNS identifiability)
        for est in ["d_total", "d_naive", "d_placebo", "d_maskmark", "pns_rollout", "drg_relevant"]:
            if est not in g or g[est].isna().all():
                continue
            aucs.append(dict(dataset=ds, estimator=est, auc_all=auc(g[est], g.label),
                             auc_ps1=auc(g2[est], g2.label), n_hops=len(g), n_ps1=len(g2)))
    df["position"] = np.where(df.label == 0, "redundant",
                              np.where(df.is_final == 1, "gold-final", "gold-nonfinal"))
    pos = df.groupby(["dataset", "position"])[["d_total", "d_naive", "d_placebo",
                                               "d_maskmark"]].mean().reset_index()
    return pd.DataFrame(aucs), pos


In [ ]:
%%writefile hgcd/report.py
"""LaTeX tables and figures written to the results directory (\\input by the paper)."""

from __future__ import annotations

import os
from typing import Dict, List

import numpy as np
import pandas as pd

from .metrics import accuracy_summary, ate_table

NAMES = {
    "direct": "Direct (no CoT)", "cot": "CoT", "sc": "Self-consistency", "cad": "CAD", "adacad": "AdaCAD",
    "cad_chain": "CAD-chain", "hgcd": "\\textbf{HGCD}", "hgcd_log": "HGCD (log-space)",
    "hgcd_naive": "HGCD (naive mask)", "hgcd_uniform": "HGCD (no graph)",
    "hgcd_placebo": "HGCD (placebo)", "hgcd_ada": "HGCD (adaptive $\\alpha$)",
    "l1+cot": "L1 + CoT", "l1+hgcd": "\\textbf{L1 + HGCD}", "free": "Free generation",
}
ORDER = ["direct", "cot", "sc", "cad", "adacad", "cad_chain", "hgcd_naive", "hgcd_uniform",
         "hgcd_log", "hgcd_ada", "hgcd_placebo", "hgcd", "l1+cot", "l1+hgcd", "free"]


def _fmt(m, lo, hi, pct=True):
    if m is None or np.isnan(m):
        return "--"
    s = 100 if pct else 1
    return f"{m * s:.1f}\\,{{\\scriptsize[{lo * s:.1f},{hi * s:.1f}]}}"


def _write(path: str, s: str):
    with open(path, "w") as f:
        f.write(s)


def main_table(df: pd.DataFrame, out: str, id_max_k: int = 4, col: str = "correct",
               by_split: bool = True) -> pd.DataFrame:
    d = df[df.cond == "orig"].copy()
    d["split"] = np.where(d.k <= id_max_k, "ID", "OOD")
    s2 = accuracy_summary(d, ["method"], col).assign(split="All")
    s = pd.concat([accuracy_summary(d, ["method", "split"], col), s2]) if by_split else s2
    splits = [x for x in ["ID", "OOD", "All"] if x in set(s.split)]
    lines = ["\\begin{tabular}{l" + "c" * len(splits) + "}", "\\toprule",
             "Method & " + " & ".join(f"{x} ($n$={int(s[s.split == x].n.max())})" for x in splits)
             + " \\\\", "\\midrule"]
    for m in [m for m in ORDER if m in set(s.method)]:
        cells = []
        for x in splits:
            r = s[(s.method == m) & (s.split == x)]
            cells.append(_fmt(*r[["mean", "lo", "hi"]].values[0]) if len(r) else "--")
        lines.append(f"{NAMES.get(m, m)} & " + " & ".join(cells) + " \\\\")
    lines += ["\\bottomrule", "\\end{tabular}"]
    _write(out, "\n".join(lines) + "\n")
    return s


def ate_latex(df: pd.DataFrame, out: str, col: str = "correct") -> pd.DataFrame:
    a = ate_table(df, col=col)
    conds = sorted(set(a.cond))
    lines = ["\\begin{tabular}{l" + "c" * len(conds) + "}", "\\toprule",
             "Method & " + " & ".join(c.replace("_", "\\_") for c in conds) + " \\\\",
             "\\midrule"]
    for m in [m for m in ORDER if m in set(a.method)]:
        cells = []
        for c in conds:
            r = a[(a.method == m) & (a.cond == c)]
            cells.append(_fmt(*r[["ate", "lo", "hi"]].values[0]) if len(r) else "--")
        lines.append(f"{NAMES.get(m, m)} & " + " & ".join(cells) + " \\\\")
    lines += ["\\bottomrule", "\\end{tabular}"]
    _write(out, "\n".join(lines) + "\n")
    return a


def estimator_latex(aucs: pd.DataFrame, pos: pd.DataFrame, out_auc: str, out_pos: str):
    en = {"d_total": "$\\Delta^{\\text{total}}$ (graph-aware mask)",
          "d_naive": "$\\Delta^{\\text{naive}}$ (mask hop only)",
          "d_placebo": "$\\Delta^{\\text{placebo}}$ (content-preserving)",
          "d_maskmark": "$\\Delta^{\\text{marker}}$ (mask token, content kept)",
          "pns_rollout": "PNS (rollout)", "drg_relevant": "DRG path indicator"}
    ds = sorted(set(aucs.dataset))
    lines = ["\\begin{tabular}{l" + "cc" * len(ds) + "}", "\\toprule",
             "Estimator & " + " & ".join(f"\\multicolumn{{2}}{{c}}{{{d}}}" for d in ds) + " \\\\",
             " & " + " & ".join(["all", "PS$=$1"] * len(ds)) + " \\\\", "\\midrule"]
    for e in ["d_naive", "d_total", "pns_rollout", "drg_relevant", "d_placebo", "d_maskmark"]:
        if e not in set(aucs.estimator):
            continue
        cells = []
        for d in ds:
            r = aucs[(aucs.dataset == d) & (aucs.estimator == e)]
            cells += ([f"{r.auc_all.values[0]:.3f}", f"{r.auc_ps1.values[0]:.3f}"]
                      if len(r) else ["--", "--"])
        lines.append(f"{en[e]} & " + " & ".join(cells) + " \\\\")
    lines += ["\\bottomrule", "\\end{tabular}"]
    _write(out_auc, "\n".join(lines) + "\n")
    lines = ["\\begin{tabular}{llcccc}", "\\toprule",
             "Dataset & Hop type & $\\Delta^{\\text{total}}$ & $\\Delta^{\\text{naive}}$ & "
             "$\\Delta^{\\text{placebo}}$ & $\\Delta^{\\text{marker}}$ \\\\", "\\midrule"]
    for _, r in pos.iterrows():
        lines.append(f"{r.dataset} & {r.position} & {r.d_total:.3f} & {r.d_naive:.3f} & "
                     f"{r.d_placebo:.3f} & {r.d_maskmark:.3f} \\\\")
    lines += ["\\bottomrule", "\\end{tabular}"]
    _write(out_pos, "\n".join(lines) + "\n")


def figures(df: pd.DataFrame, sweep: pd.DataFrame, outdir: str, prefix: str = "clutrr"):
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    d = df[df.cond == "orig"]
    fig, ax = plt.subplots(figsize=(4.2, 3.0))
    for m in ["cot", "cad_chain", "hgcd_naive", "hgcd"]:
        g = d[d.method == m].groupby("k").correct.mean()
        if len(g):
            ax.plot(g.index, g.values * 100, marker="o", label=NAMES[m].replace("\\textbf{", "").replace("}", ""))
    ax.set_xlabel("number of hops $k$")
    ax.set_ylabel("accuracy (%)")
    ax.legend(fontsize=7, frameon=False)
    fig.tight_layout()
    fig.savefig(os.path.join(outdir, f"{prefix}_acc_vs_k.pdf"))
    plt.close(fig)
    if sweep is not None and len(sweep):
        fig, ax = plt.subplots(figsize=(4.2, 3.0))
        for m, g in sweep.groupby("method"):
            ax.plot(g.alpha, g.acc * 100, marker=".", label=NAMES.get(m, m).replace("\\textbf{", "").replace("}", ""))
        ax.set_xscale("symlog", linthresh=0.1)
        ax.set_xlabel("$\\alpha$")
        ax.set_ylabel("accuracy (%)")
        ax.legend(fontsize=7, frameon=False)
        fig.tight_layout()
        fig.savefig(os.path.join(outdir, f"{prefix}_alpha_sweep.pdf"))
        plt.close(fig)


In [ ]:
%%writefile hgcd/pipeline.py
"""End-to-end pipeline used by the Kaggle notebook and the CLI.

    python -m hgcd.pipeline --preset smoke      # CPU sanity run (minutes)
    python -m hgcd.pipeline --preset kaggle     # 1x T4, Qwen2.5-1.5B-Instruct (~2-4 h)
    python -m hgcd.pipeline --preset a100       # 1x A100, 7B model
"""

from __future__ import annotations

import argparse
import json
import os
import pickle
import time
from dataclasses import asdict, dataclass, field
from typing import List

import numpy as np
import pandas as pd

from . import data as D
from . import experiments as E
from . import metrics as M
from . import report as R


@dataclass
class Config:
    model: str = "Qwen/Qwen2.5-1.5B-Instruct"
    outdir: str = "results/run"
    seed: int = 0
    batch_size: int = 8
    # CLUTRR decoding
    ks_test: List[int] = field(default_factory=lambda: [2, 3, 4, 5, 6, 7])
    n_per_k: int = 40
    n_dev: int = 60                      # validation split (k in 2..4) for alpha selection
    conditions: List[str] = field(default_factory=lambda: ["orig", "rename", "order",
                                                           "distract", "placebo_input"])
    l1: bool = True
    l1_alpha: float = 1.0
    sc_n: int = 5                        # self-consistency samples (0 disables)
    alphas: List[float] = field(default_factory=lambda: [0.0, 0.25, 0.5, 1.0, 2.0, 4.0])
    beta: float = 0.1
    # estimator validation (gold chains + inserted redundant hop)
    ks_est: List[int] = field(default_factory=lambda: [2, 3, 4])
    n_est_per_k: int = 30
    est_rollouts: int = 0
    # MuSiQue
    n_ms: int = 120
    ms_conditions: List[str] = field(default_factory=lambda: ["orig", "dist8", "order"])
    n_est_ms: int = 80
    run_musique: bool = True


PRESETS = {
    "smoke": dict(model="Qwen/Qwen2.5-0.5B-Instruct", ks_test=[2, 3, 5], n_per_k=4, n_dev=6,
                  conditions=["orig", "rename", "distract"], l1=True, sc_n=3, ks_est=[2, 3],
                  n_est_per_k=4, est_rollouts=0, n_ms=6, ms_conditions=["orig", "dist8"],
                  n_est_ms=6, batch_size=4, alphas=[0.0, 0.5, 1.0, 2.0]),
    "kaggle": dict(),
    "kaggle_full": dict(n_per_k=80, n_dev=120, n_est_per_k=60, est_rollouts=4, n_ms=300,
                        n_est_ms=200),
    "a100": dict(model="Qwen/Qwen2.5-7B-Instruct", ks_test=[2, 3, 4, 5, 6, 7, 8, 9, 10],
                 n_per_k=100, n_dev=150, n_est_per_k=80, est_rollouts=4, n_ms=500,
                 n_est_ms=300, batch_size=16),
}


def _log(m):
    print(time.strftime("[%H:%M:%S] "), m, flush=True)


def select_alpha(recs, methods, alphas, beta, table_fn):
    rows = []
    for a in alphas:
        df = table_fn(recs, alpha=a, beta=beta, methods=methods)
        for m, g in df.groupby("method"):
            rows.append(dict(method=m, alpha=a, acc=g.correct.mean()))
    sweep = pd.DataFrame(rows)
    best = {m: float(g.sort_values(["acc", "alpha"], ascending=[False, True]).alpha.iloc[0])
            for m, g in sweep.groupby("method")}
    return best, sweep


def _table_with_alphas(recs, best, beta, table_fn):
    parts = []
    for m, a in best.items():
        parts.append(table_fn(recs, alpha=a, beta=beta, methods=[m.replace("l1+", "")]))
    df = pd.concat(parts)
    keep = set(best)
    return df[df.method.isin(keep)].drop_duplicates()


def run(cfg: Config):
    from .lm import LM
    os.makedirs(cfg.outdir, exist_ok=True)
    with open(os.path.join(cfg.outdir, "config.json"), "w") as f:
        json.dump(asdict(cfg), f, indent=1)
    t0 = time.time()
    lm = LM(cfg.model)
    _log(f"loaded {cfg.model} on {lm.device}")
    summary = {"model": cfg.model}

    # ------------------------------------------------------------- data
    train = D.load_clutrr("train")
    test = D.load_clutrr("test")
    dev = D.load_clutrr("validation")
    rels = D.clutrr_relations(train + test)
    pool = train
    summary["clutrr_filter"] = {"train": len(train), "test": len(test), "dev": len(dev)}

    # ------------------------------------------------------------- 1) estimator validation
    est_ex = D.sample_by_k(test, cfg.ks_est, cfg.n_est_per_k, cfg.seed + 7)
    _log(f"estimator validation on {len(est_ex)} CLUTRR gold chains")
    est_rows = E.run_estimators_clutrr(lm, est_ex, pool, rels, rollouts=cfg.est_rollouts,
                                       batch_size=cfg.batch_size, seed=cfg.seed)
    ms = None
    if cfg.run_musique:
        ms = D.load_musique("dev")
        rng = np.random.default_rng(cfg.seed)
        ms_est = [ms[i] for i in rng.choice(len(ms), cfg.n_est_ms, replace=False)]
        _log(f"estimator validation on {len(ms_est)} MuSiQue gold chains")
        est_rows += E.run_estimators_musique(lm, ms_est, seed=cfg.seed)
    pd.DataFrame(est_rows).to_csv(os.path.join(cfg.outdir, "estimators.csv"), index=False)
    aucs, pos = M.estimator_table(est_rows)
    R.estimator_latex(aucs, pos, os.path.join(cfg.outdir, "tab_estimators_auc.tex"),
                      os.path.join(cfg.outdir, "tab_estimators_pos.tex"))
    summary["estimator_auc"] = aucs.to_dict("records")
    summary["estimator_position"] = pos.to_dict("records")
    _log("\n" + aucs.to_string())
    _log("\n" + pos.to_string())

    # ------------------------------------------------------------- 2) alpha selection on dev
    dev_ex = D.sample_by_k(dev, [2, 3, 4], max(1, cfg.n_dev // 3), cfg.seed + 1)
    _log(f"alpha selection on {len(dev_ex)} CLUTRR validation examples")
    dev_recs = E.run_clutrr(lm, dev_ex, pool, rels, conditions=["orig"],
                            batch_size=cfg.batch_size, l1=False)
    best, sweep = select_alpha(dev_recs, M.GREEDY_METHODS, cfg.alphas, cfg.beta, M.clutrr_table)
    best["l1+cot"], best["l1+hgcd"] = 0.0, best.get("hgcd", 1.0)
    sweep.to_csv(os.path.join(cfg.outdir, "alpha_sweep_dev.csv"), index=False)
    summary["alpha"] = best
    _log(f"selected alphas: {best}")

    # ------------------------------------------------------------- 3) CLUTRR test
    test_ex = D.sample_by_k(test, cfg.ks_test, cfg.n_per_k, cfg.seed)
    _log(f"CLUTRR test: {len(test_ex)} examples x {len(cfg.conditions)} conditions")
    recs = E.run_clutrr(lm, test_ex, pool, rels, conditions=cfg.conditions,
                        batch_size=cfg.batch_size, l1=cfg.l1, l1_alpha=cfg.l1_alpha,
                        sc_n=cfg.sc_n)
    with open(os.path.join(cfg.outdir, "clutrr_records.pkl"), "wb") as f:
        pickle.dump(recs, f)
    df = _table_with_alphas(recs, best, cfg.beta, M.clutrr_table)
    df.to_csv(os.path.join(cfg.outdir, "clutrr_decisions.csv"), index=False)
    s = R.main_table(df, os.path.join(cfg.outdir, "tab_clutrr_main.tex"))
    a = R.ate_latex(df[~df.method.str.startswith("l1+")],
                    os.path.join(cfg.outdir, "tab_clutrr_ate.tex"))
    R.figures(df, sweep, cfg.outdir, "clutrr")
    comps = [M.paired_compare(df, "hgcd", b) for b in ["cot", "sc", "cad_chain", "hgcd_naive",
                                                        "hgcd_uniform", "hgcd_log"]]
    if cfg.l1:
        comps.append(M.paired_compare(df, "l1+hgcd", "cot"))
    summary["clutrr_acc"] = s.to_dict("records")
    summary["clutrr_ate"] = a.to_dict("records")
    summary["clutrr_paired"] = comps
    o = df[df.cond == "orig"]
    summary["clutrr_format"] = {"fmt_ok": float(o.fmt_ok.mean()), "hop_fid": float(o.hop_fid.mean()),
                                "drg_connected": float(o.drg_connected.mean()),
                                "mean_hops": float(o.n_hops.mean())}
    _log("\n" + s.to_string())
    _log("\n" + pd.DataFrame(comps).to_string())

    # ------------------------------------------------------------- 4) MuSiQue
    if cfg.run_musique:
        rng = np.random.default_rng(cfg.seed + 3)
        ms_ex = [ms[i] for i in rng.choice(len(ms), cfg.n_ms, replace=False)]
        _log(f"MuSiQue: {len(ms_ex)} examples x {len(cfg.ms_conditions)} conditions")
        mrecs = E.run_musique(lm, ms_ex, conditions=cfg.ms_conditions,
                              batch_size=max(1, cfg.batch_size // 2))
        with open(os.path.join(cfg.outdir, "musique_records.pkl"), "wb") as f:
            pickle.dump(mrecs, f)
        mbest = {m: best.get(m, 1.0) for m in M.GREEDY_METHODS}
        mdf = _table_with_alphas(mrecs, mbest, cfg.beta, M.musique_table)
        free = M.musique_table(mrecs, methods=["cot"])
        mdf = pd.concat([mdf, free[free.method == "free"]])
        mdf.to_csv(os.path.join(cfg.outdir, "musique_decisions.csv"), index=False)
        ms_s = R.main_table(mdf, os.path.join(cfg.outdir, "tab_musique_main.tex"), col="em",
                            by_split=False)
        ms_a = R.ate_latex(mdf, os.path.join(cfg.outdir, "tab_musique_ate.tex"), col="em")
        summary["musique_em"] = ms_s.to_dict("records")
        summary["musique_ate"] = ms_a.to_dict("records")
        summary["musique_cand_recall"] = float(mdf[mdf.cond == "orig"].cand_recall.mean())
        summary["musique_paired"] = [M.paired_compare(mdf, "hgcd", b, col="em")
                                     for b in ["cot", "cad", "cad_chain", "hgcd_naive"]]
        _log("\n" + ms_s.to_string())

    summary["boundary_fallbacks"] = lm.n_boundary_fallback
    summary["minutes"] = (time.time() - t0) / 60
    with open(os.path.join(cfg.outdir, "summary.json"), "w") as f:
        json.dump(summary, f, indent=1, default=float)
    _log(f"done in {summary['minutes']:.1f} min -> {cfg.outdir}")
    return summary


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--preset", default="smoke", choices=list(PRESETS))
    ap.add_argument("--model", default=None)
    ap.add_argument("--outdir", default=None)
    args = ap.parse_args()
    cfg = Config(**PRESETS[args.preset])
    if args.model:
        cfg.model = args.model
    cfg.outdir = args.outdir or f"results/{args.preset}"
    run(cfg)


if __name__ == "__main__":
    main()


In [ ]:
import os
# ---- configuration -------------------------------------------------------------
# PRESET: "kaggle" (~2-4 h on one T4), "kaggle_full" (larger n, ~8-10 h), "smoke" (minutes)
PRESET = "kaggle"
MAIN_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
SECOND_MODEL = "HuggingFaceTB/SmolLM2-1.7B-Instruct"   # different model family, ungated
RUN_SECOND_ON_GPU1 = True
OUT = "/kaggle/working/results" if os.path.isdir("/kaggle/working") else "results"


## Optional: second model family on GPU 1 (runs in the background)
Log: `results/<preset>_second.log`. Check it at the end of the notebook.

In [ ]:
import subprocess, sys, torch
proc = None
if RUN_SECOND_ON_GPU1 and torch.cuda.device_count() > 1:
    os.makedirs(OUT, exist_ok=True)
    env = dict(os.environ, CUDA_VISIBLE_DEVICES='1')
    log = open(f'{OUT}/{PRESET}_second.log', 'w')
    proc = subprocess.Popen([sys.executable, '-m', 'hgcd.pipeline', '--preset', PRESET,
                             '--model', SECOND_MODEL, '--outdir', f'{OUT}/{PRESET}_second'],
                            env=env, stdout=log, stderr=subprocess.STDOUT)
    print('second model started, pid', proc.pid)
else:
    print('single GPU: skipping the second model')

## Main run (GPU 0)

In [ ]:
from hgcd.pipeline import Config, PRESETS, run
cfg = Config(**PRESETS[PRESET])
cfg.model = MAIN_MODEL if PRESET != 'smoke' else cfg.model
cfg.outdir = f'{OUT}/{PRESET}_main'
summary = run(cfg)

## Results

In [ ]:
import json, glob, pandas as pd
from IPython.display import display, IFrame
d = cfg.outdir
print(json.dumps({k: summary[k] for k in ['alpha', 'clutrr_format', 'minutes'] if k in summary}, indent=1))
for f in ['tab_estimators_auc.tex', 'tab_estimators_pos.tex', 'tab_clutrr_main.tex',
          'tab_clutrr_ate.tex', 'tab_musique_main.tex', 'tab_musique_ate.tex']:
    p = os.path.join(d, f)
    if os.path.exists(p):
        print('=' * 20, f); print(open(p).read())
display(pd.DataFrame(summary.get('clutrr_paired', [])))
display(pd.DataFrame(summary.get('musique_paired', [])))

In [ ]:
import matplotlib.pyplot as plt
df = pd.read_csv(os.path.join(d, 'clutrr_decisions.csv'))
o = df[df.cond == 'orig']
ax = o[o.method.isin(['cot', 'cad_chain', 'hgcd_naive', 'hgcd'])].groupby(['k', 'method']).correct.mean().unstack().plot(marker='o')
ax.set_ylabel('accuracy'); plt.show()

## Wait for the GPU-1 run and package everything

In [ ]:
if proc is not None:
    proc.wait()
    print(open(f'{OUT}/{PRESET}_second.log').read()[-3000:])
import shutil
shutil.make_archive(f'{OUT}_{PRESET}', 'zip', OUT)
print('download:', f'{OUT}_{PRESET}.zip')